# Handling Edge Cases & Candidate Distribution Shift
### Sprint 2, Item 5

Tests the matching pipeline against inputs that don't look like the clean training
data: empty text, very short resumes, unusual characters, categories with no matching
job description, and resumes that are unusually different from the rest of the cohort
(distribution shift). Wraps the scoring logic so it fails gracefully instead of
crashing on real-world messy input.


In [1]:
%run config.ipynb

Config loaded. Data will be stored under: c:\Users\RanVic\Downloads\applicant_screening_sprint2\applicant_screening_nb\data_pipeline\data


## 1. Load model + data

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

resumes_clean = pd.read_csv(PROCESSED_RESUMES_CSV)
job_descriptions = pd.read_csv(JOB_DESCRIPTIONS_CSV)

model = SentenceTransformer("all-MiniLM-L6-v2")
job_embeddings = model.encode(job_descriptions["job_description"].tolist(), normalize_embeddings=True)

print("Loaded model and job embeddings.")

C:\Users\RanVic\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 976.10it/s]


Loaded model and job embeddings.


## 2. A robust scoring function
Wraps embedding + similarity scoring with input validation, so bad input returns a
clear error/warning instead of an unhandled exception.

In [3]:
from dataclasses import dataclass

@dataclass
class ScoreResult:
    success: bool
    best_category: str | None = None
    similarity: float | None = None
    warning: str | None = None

MIN_MEANINGFUL_LENGTH = 30  # shorter than this, treat as too little signal to score reliably

def safe_score_resume(text: str) -> ScoreResult:
    if not isinstance(text, str) or text.strip() == "":
        return ScoreResult(success=False, warning="Empty or non-text input — cannot score.")

    cleaned = text.strip()
    if len(cleaned) < MIN_MEANINGFUL_LENGTH:
        return ScoreResult(
            success=False,
            warning=f"Input is only {len(cleaned)} characters — too short for a reliable match. "
                    f"Flag for manual review instead of automatic scoring.",
        )

    try:
        embedding = model.encode([cleaned], normalize_embeddings=True)[0]
    except Exception as e:
        return ScoreResult(success=False, warning=f"Embedding failed: {e}")

    sims = job_embeddings @ embedding
    best_idx = int(np.argmax(sims))
    return ScoreResult(
        success=True,
        best_category=job_descriptions.iloc[best_idx][COL_CATEGORY],
        similarity=float(sims[best_idx]),
    )

## 3. Test against edge cases

In [4]:
edge_cases = {
    "empty_string": "",
    "whitespace_only": "    \n\t  ",
    "none_value": None,
    "very_short": "Nurse.",
    "non_english_snippet": "Ingeniero de software con experiencia en Python y machine learning",
    "special_characters_only": "!!!@@@###$$$%%%^^^&&&***",
    "extremely_long": "Experienced software engineer. " * 2000,  # ~10k+ words
    "normal_resume": resumes_clean.iloc[0]["clean_text"],
}

for name, text in edge_cases.items():
    result = safe_score_resume(text)
    if result.success:
        print(f"[{name}] -> matched '{result.best_category}' (similarity={result.similarity:.3f})")
    else:
        print(f"[{name}] -> SKIPPED: {result.warning}")

[empty_string] -> SKIPPED: Empty or non-text input — cannot score.
[whitespace_only] -> SKIPPED: Empty or non-text input — cannot score.
[none_value] -> SKIPPED: Empty or non-text input — cannot score.
[very_short] -> SKIPPED: Input is only 6 characters — too short for a reliable match. Flag for manual review instead of automatic scoring.
[non_english_snippet] -> matched 'ENGINEERING' (similarity=0.271)
[special_characters_only] -> SKIPPED: Input is only 24 characters — too short for a reliable match. Flag for manual review instead of automatic scoring.
[extremely_long] -> matched 'ENGINEERING' (similarity=0.463)
[normal_resume] -> matched 'HR' (similarity=0.504)


## 4. Distribution shift: does an out-of-scope resume still get force-matched?
A resume from a job category that has **no** corresponding job description in our
24 categories should still get *some* similarity score (since cosine similarity
always returns a number) — but that top match may be meaningless. This is the real
risk with distribution shift: the pipeline doesn't know when it's being asked
something outside its coverage.

In [5]:
out_of_scope_resume = (
    "Professional saxophonist with 10 years experience performing jazz concerts, "
    "touring internationally, and teaching music theory at a conservatory."
)

result = safe_score_resume(out_of_scope_resume)
print(f"Out-of-scope resume matched: '{result.best_category}' (similarity={result.similarity:.3f})")
print()
print("This score is misleadingly confident-looking, but the true category (Musician)")
print("simply doesn't exist in our 24 job categories -- the model still returns its")
print("best available guess, not an 'I don't know'.")

Out-of-scope resume matched: 'ARTS' (similarity=0.317)

This score is misleadingly confident-looking, but the true category (Musician)
simply doesn't exist in our 24 job categories -- the model still returns its
best available guess, not an 'I don't know'.


## 5. Mitigation: a similarity floor
One practical guardrail: define a minimum similarity threshold below which a match
is flagged as "no confident category" instead of being silently force-matched.

In [6]:
SIMILARITY_FLOOR = 0.35  # tune this against validation data; illustrative default here

def safe_score_resume_with_floor(text: str) -> ScoreResult:
    result = safe_score_resume(text)
    if result.success and result.similarity is not None and result.similarity < SIMILARITY_FLOOR:
        return ScoreResult(
            success=False,
            warning=f"Best match '{result.best_category}' scored only "
                    f"{result.similarity:.3f}, below the {SIMILARITY_FLOOR} confidence floor — "
                    f"flagging as no confident category rather than force-matching.",
        )
    return result

result = safe_score_resume_with_floor(out_of_scope_resume)
print(f"With confidence floor: success={result.success}, warning={result.warning}")

With confidence floor: success=False, warning=Best match 'ARTS' scored only 0.317, below the 0.35 confidence floor — flagging as no confident category rather than force-matching.


---
## Notes
- `SIMILARITY_FLOOR = 0.35` is illustrative — the right threshold should be tuned
  against real labeled examples of "no good match" cases, which this project doesn't
  currently have. Document this as a known limitation, not a solved problem.
- These checks (empty/short input, similarity floor) are the minimum viable guardrails
  for a screening tool that could otherwise silently assign confident-looking but
  meaningless categories to resumes it has no real way to classify.
